# 🫘 Chronic Kidney Disease — EDA & Classification

> **Dataset:** UCI Machine Learning Repository — Chronic Kidney Disease  
> **Records:** 400 patients | **Features:** 25 | **Target:** `ckd` / `notckd`

---

## 📋 Table of Contents
1. [Import Libraries](#1-import-libraries)
2. [Load Dataset](#2-load-dataset)
3. [Initial Exploration](#3-initial-exploration)
4. [Data Cleaning](#4-data-cleaning)
5. [Missing Value Analysis](#5-missing-value-analysis)
6. [Imputation](#6-imputation)
7. [Exploratory Data Analysis](#7-exploratory-data-analysis)
8. [Correlation Heatmap](#8-correlation-heatmap)
9. [Label Encoding & Feature Split](#9-label-encoding--feature-split)
10. [Model Training & Evaluation](#10-model-training--evaluation)
11. [Model Comparison](#11-model-comparison)
12. [Key Findings](#12-key-findings)

---
## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# Plot style
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 100

print('✅ All libraries imported successfully.')

---
## 2. Load Dataset

In [ ]:
df = pd.read_csv('kidney_disease.csv')
print(f'Shape: {df.shape}')
df.head()

---
## 3. Initial Exploration

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print('Column dtypes:\n')
print(df.dtypes)

In [ ]:
# Target class distribution
fig, ax = plt.subplots(figsize=(6, 4))
counts = df['classification'].value_counts()
colors = ['#6b2d6b', '#c06040']
bars = ax.bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.5, width=0.5)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            f'{val}\n({val/len(df)*100:.1f}%)', ha='center', fontsize=11, fontweight='bold')
ax.set_title('Target Class Distribution', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Classification')
ax.set_ylabel('Count')
ax.set_ylim(0, max(counts.values) + 40)
sns.despine()
plt.tight_layout()
plt.show()

---
## 4. Data Cleaning

Several columns have dirty string values — extra whitespace, tab characters — and some numerical columns were incorrectly stored as `object` dtype.

In [ ]:
# Convert pcv, wc, rc to numeric (stored as object due to mixed values)
for col in ['pcv', 'wc', 'rc']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print('Converted pcv, wc, rc → float64')

In [ ]:
# Check dirty values before cleaning
print('dm unique values before:', df['dm'].unique())
print('cad unique values before:', df['cad'].unique())
print('classification unique values before:', df['classification'].unique())

In [ ]:
# Fix whitespace / tab issues
df['dm']             = df['dm'].replace({'\tno': 'no', '\tyes': 'yes', ' yes': 'yes'})
df['cad']            = df['cad'].replace('\tno', 'no')
df['classification'] = df['classification'].replace('ckd\t', 'ckd')

print('After cleaning:')
print('dm unique values:            ', df['dm'].unique())
print('cad unique values:           ', df['cad'].unique())
print('classification unique values:', df['classification'].unique())

In [ ]:
# Separate column types
cat_cols = [col for col in df.columns if df[col].dtype == 'object']
num_cols = [col for col in df.columns if df[col].dtype != 'object']

print(f'Categorical columns ({len(cat_cols)}): {cat_cols}')
print(f'\nNumerical columns ({len(num_cols)}): {num_cols}')

In [ ]:
# Unique values in each categorical column
for col in cat_cols:
    print(f"  {col:20s} → {df[col].unique()}")

---
## 5. Missing Value Analysis

In [ ]:
total_missing = df.isnull().sum().sum()
print(f'Total missing values: {total_missing} out of {df.shape[0] * df.shape[1]} cells ({total_missing / (df.shape[0] * df.shape[1]) * 100:.1f}%)\n')
missing = df.isnull().sum().sort_values(ascending=False)
missing[missing > 0]

In [ ]:
# Missing value heatmap
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart
mv = df.isnull().sum().sort_values(ascending=False)
mv = mv[mv > 0]
colors_bar = ['#c0392b' if v > 80 else '#e67e22' if v > 30 else '#f1c40f' for v in mv.values]
axes[0].barh(mv.index, mv.values, color=colors_bar, edgecolor='white')
axes[0].set_title('Missing Values per Column', fontweight='bold')
axes[0].set_xlabel('Number of Missing Values')
for i, v in enumerate(mv.values):
    axes[0].text(v + 1, i, f'{v} ({v/400*100:.0f}%)', va='center', fontsize=9)
axes[0].invert_yaxis()
sns.despine(ax=axes[0])

# Heatmap of nulls
missing_cols = mv.index.tolist()
sns.heatmap(df[missing_cols].isnull().T, cmap='YlOrRd', cbar=False,
            yticklabels=missing_cols, xticklabels=False, ax=axes[1])
axes[1].set_title('Missing Value Pattern (each column)', fontweight='bold')
axes[1].set_xlabel('Patients (0–399)')

plt.suptitle('Missing Data Overview', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 6. Imputation

- **Numerical columns** → Random sampling from non-null values (preserves distribution shape)
- **`rbc`, `pc`** (high-missingness categorical) → Random sampling
- **Remaining categorical** → Mode imputation

In [ ]:
def random_value_imputation(feature):
    """Fill NaN with a random sample drawn from the non-null values."""
    random_sample = df[feature].dropna().sample(df[feature].isna().sum(), random_state=42)
    random_sample.index = df[df[feature].isnull()].index
    df.loc[df[feature].isnull(), feature] = random_sample

def impute_mode(feature):
    """Fill NaN with the column mode."""
    df[feature] = df[feature].fillna(df[feature].mode()[0])

# Apply imputation
for col in num_cols:
    random_value_imputation(col)

random_value_imputation('rbc')
random_value_imputation('pc')

for col in cat_cols:
    impute_mode(col)

print(f'Missing values after imputation: {df.isnull().sum().sum()}')

---
## 7. Exploratory Data Analysis

### 7.1 Numerical Feature Distributions

In [ ]:
plot_cols = [c for c in num_cols if c != 'id']
n = len(plot_cols)
ncols = 4
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 3.5))
axes = axes.flatten()

for i, col in enumerate(plot_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color='#6b2d6b', alpha=0.7)
    axes[i].set_title(col, fontweight='bold', fontsize=12)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Count')
    sns.despine(ax=axes[i])

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numerical Feature Distributions', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 7.2 Categorical Feature Counts

In [ ]:
plot_cat = [c for c in cat_cols if c != 'classification']
ncols = 3
nrows = (len(plot_cat) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3.5))
axes = axes.flatten()

for i, col in enumerate(plot_cat):
    sns.countplot(x=df[col], ax=axes[i], palette='rocket', edgecolor='white')
    axes[i].set_title(f'{col}', fontweight='bold', fontsize=12)
    axes[i].set_xlabel('')
    total = len(df[col].dropna())
    for p in axes[i].patches:
        axes[i].annotate(f'{p.get_height()}\n({p.get_height()/total*100:.0f}%)',
                         (p.get_x() + p.get_width() / 2., p.get_height()),
                         ha='center', va='bottom', fontsize=9)
    sns.despine(ax=axes[i])

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Categorical Feature Counts', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 7.3 Categorical Features vs Target (CKD Classification)

In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3.5))
axes = axes.flatten()

for i, col in enumerate(plot_cat):
    sns.countplot(x=col, hue='classification', data=df, ax=axes[i],
                  palette='rocket', edgecolor='white')
    axes[i].set_title(f'{col} vs classification', fontweight='bold', fontsize=11)
    axes[i].set_xlabel('')
    axes[i].legend(title='', fontsize=8)
    sns.despine(ax=axes[i])

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Categorical Features vs CKD Classification', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 7.4 Numerical Features vs Target — Box Plots

In [ ]:
ncols = 4
nrows = (len(plot_cols) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 3.5))
axes = axes.flatten()

for i, col in enumerate(plot_cols):
    sns.boxplot(x='classification', y=col, data=df,
                palette=['#6b2d6b', '#c06040'], ax=axes[i], width=0.5)
    axes[i].set_title(col, fontweight='bold', fontsize=12)
    axes[i].set_xlabel('')
    sns.despine(ax=axes[i])

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numerical Features vs CKD Classification', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 8. Correlation Heatmap

In [ ]:
plt.figure(figsize=(16, 10))
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))   # upper triangle mask

sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    linewidths=0.5,
    linecolor='white',
    square=True,
    vmin=-1, vmax=1,
    annot_kws={'size': 8}
)
plt.title('Correlation Heatmap — Numerical Features', fontsize=15, fontweight='bold', pad=14)
plt.tight_layout()
plt.show()

# Top absolute correlations
print('\nTop 10 strongest correlations (absolute value):')
corr_pairs = corr.abs().unstack()
corr_pairs = corr_pairs[corr_pairs < 1].sort_values(ascending=False).drop_duplicates()
print(corr_pairs.head(10).to_string())

---
## 9. Label Encoding & Feature Split

In [ ]:
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

print('Label encoding complete. Sample:')
df.head(3)

In [ ]:
X = df.drop(['classification', 'id'], axis=1)
y = df['classification']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=0
)

print(f'Training set : {X_train.shape[0]} samples')
print(f'Test set     : {X_test.shape[0]} samples')
print(f'Features     : {X_train.shape[1]}')

---
## 10. Model Training & Evaluation

### 10.1 Logistic Regression

In [ ]:
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)
y_pred_lr = log_reg.predict(X_test)
lg_accuracy = accuracy_score(y_test, y_pred_lr)

print(f'Logistic Regression Accuracy: {lg_accuracy:.4f} ({lg_accuracy*100:.2f}%)\n')
print(classification_report(y_test, y_pred_lr, target_names=['CKD', 'Not CKD']))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_lr),
                       display_labels=['CKD', 'Not CKD']).plot(ax=ax, colorbar=False, cmap='Purples')
ax.set_title('Logistic Regression — Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

### 10.2 Decision Tree

In [ ]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
dt_accuracy = accuracy_score(y_test, y_pred_dt)

print(f'Decision Tree Accuracy: {dt_accuracy:.4f} ({dt_accuracy*100:.2f}%)\n')
print(classification_report(y_test, y_pred_dt, target_names=['CKD', 'Not CKD']))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_dt),
                       display_labels=['CKD', 'Not CKD']).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Decision Tree — Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

### 10.3 Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
rf_accuracy = accuracy_score(y_test, y_pred_rf)

print(f'Random Forest Accuracy: {rf_accuracy:.4f} ({rf_accuracy*100:.2f}%)\n')
print(classification_report(y_test, y_pred_rf, target_names=['CKD', 'Not CKD']))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_rf),
                       display_labels=['CKD', 'Not CKD']).plot(ax=ax, colorbar=False, cmap='Greens')
ax.set_title('Random Forest — Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

#### Feature Importance — Random Forest

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#c0392b' if v > importances.quantile(0.75) else '#6b2d6b' for v in importances.values]
importances.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.set_title('Random Forest — Feature Importances', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.axvline(importances.mean(), color='gray', linestyle='--', linewidth=1, label=f'Mean ({importances.mean():.3f})')
ax.legend()
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

### 10.4 Support Vector Machine (Linear Kernel)

In [ ]:
svm = SVC(kernel='linear', C=1, random_state=42)
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)
svm_accuracy = accuracy_score(y_test, y_pred_svm)

print(f'SVM Accuracy: {svm_accuracy:.4f} ({svm_accuracy*100:.2f}%)\n')
print(classification_report(y_test, y_pred_svm, target_names=['CKD', 'Not CKD']))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_svm),
                       display_labels=['CKD', 'Not CKD']).plot(ax=ax, colorbar=False, cmap='Oranges')
ax.set_title('SVM — Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

### 10.5 K-Nearest Neighbours (k=5)

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)
knn_accuracy = accuracy_score(y_test, y_pred_knn)

print(f'KNN Accuracy: {knn_accuracy:.4f} ({knn_accuracy*100:.2f}%)\n')
print(classification_report(y_test, y_pred_knn, target_names=['CKD', 'Not CKD']))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_knn),
                       display_labels=['CKD', 'Not CKD']).plot(ax=ax, colorbar=False, cmap='Reds')
ax.set_title('KNN — Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 11. Model Comparison

In [ ]:
models_df = pd.DataFrame({
    'Model': [
        'Random Forest',
        'Decision Tree',
        'SVM (Linear)',
        'Logistic Regression',
        'KNN (k=5)'
    ],
    'Accuracy': [
        rf_accuracy,
        dt_accuracy,
        svm_accuracy,
        lg_accuracy,
        knn_accuracy
    ]
}).sort_values('Accuracy', ascending=False).reset_index(drop=True)

models_df['Accuracy (%)'] = (models_df['Accuracy'] * 100).round(2)
models_df['Rank'] = models_df.index + 1
models_df[['Rank', 'Model', 'Accuracy (%)']].style.background_gradient(subset=['Accuracy (%)'], cmap='Greens')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
palette = ['#2e7d32', '#1565c0', '#4a148c', '#e65100', '#b71c1c']
bars = axes[0].barh(
    models_df['Model'],
    models_df['Accuracy (%)'],
    color=palette,
    edgecolor='white',
    height=0.55
)
axes[0].set_xlim(60, 100)
axes[0].set_xlabel('Accuracy (%)')
axes[0].set_title('Model Accuracy Comparison', fontsize=13, fontweight='bold')
axes[0].invert_yaxis()
for bar, val in zip(bars, models_df['Accuracy (%)']):
    axes[0].text(val + 0.2, bar.get_y() + bar.get_height()/2,
                 f'{val:.2f}%', va='center', fontsize=10, fontweight='bold')
axes[0].axvline(90, color='gray', linestyle='--', linewidth=1, alpha=0.6, label='90% threshold')
axes[0].legend(fontsize=9)
sns.despine(ax=axes[0])

# All confusion matrices side by side
all_preds = [y_pred_rf, y_pred_dt, y_pred_svm, y_pred_lr, y_pred_knn]
all_names = ['Random Forest', 'Decision Tree', 'SVM', 'Logistic Reg', 'KNN']
all_colors = ['Greens', 'Blues', 'Oranges', 'Purples', 'Reds']

axes[1].axis('off')
fig2, axes2 = plt.subplots(1, 5, figsize=(20, 3.5))
for i, (pred, name, cmap) in enumerate(zip(all_preds, all_names, all_colors)):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=axes2[i],
                xticklabels=['CKD', 'NotCKD'], yticklabels=['CKD', 'NotCKD'],
                cbar=False, linewidths=1, linecolor='white')
    acc = accuracy_score(y_test, pred)
    axes2[i].set_title(f'{name}\n{acc*100:.1f}%', fontsize=10, fontweight='bold')
    axes2[i].set_xlabel('Predicted')
    axes2[i].set_ylabel('Actual' if i == 0 else '')

plt.suptitle('All Models — Confusion Matrices', fontsize=13, fontweight='bold', y=1.04)
plt.tight_layout()
plt.show()

fig.tight_layout()
plt.show()

---
## 12. Key Findings

| # | Finding |
|---|---|
| 🥇 | **Random Forest** is the best model at **95.83% accuracy** with **perfect recall on CKD** (0 missed cases) |
| 🔬 | **Serum creatinine (`sc`), blood urea (`bu`), and haemoglobin (`hemo`)** are the strongest predictors |
| ⚕️ | **Hypertension (`htn`)** is almost exclusively present in CKD patients in this dataset |
| 📊 | **Strong correlation: `su ↔ bgr` (0.72)** — blood sugar levels track closely with blood glucose |
| 📉 | **Negative correlation: `sc ↔ sod` (−0.69)** — as creatinine rises, sodium drops (classic CKD sign) |
| ⚠️ | **KNN performs poorly (70.8%)** due to mixed feature scales — normalisation would improve this |
| 🧹 | **25.2% of data was missing** — imputed using random sampling to preserve original distributions |
| ⚖️ | **Class imbalance (62.5% CKD / 37.5% Not-CKD)** is moderate and models handle it without resampling |

In [ ]:
print('=' * 55)
print('   CHRONIC KIDNEY DISEASE — FINAL SUMMARY')
print('=' * 55)
print(f'  Dataset      : 400 patients, 25 features')
print(f'  CKD / NotCKD : 250 / 150')
print(f'  Train / Test : {len(X_train)} / {len(X_test)}')
print()
print('  Model Results:')
for _, row in models_df.iterrows():
    marker = '  ← BEST ✅' if row['Rank'] == 1 else ''
    print(f"    {int(row['Rank'])}. {row['Model']:<25} {row['Accuracy (%)']:.2f}%{marker}")
print('=' * 55)